### Tumor Segmentation Notebook

#### Load Librairies

In [1]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm

from mobile_unetr import MobileUNETR


#### Defining Training Hyperparameters 

In [2]:
# Data and training configuration
DATA_ROOT = Path('/Users/felixdelisle/Desktop/models/datasets/brisc2025/segmentation_task')
TRAIN_IMAGES = DATA_ROOT / 'train' / 'images'
TRAIN_MASKS = DATA_ROOT / 'train' / 'masks'
VAL_IMAGES = DATA_ROOT / 'test' / 'images'
VAL_MASKS = DATA_ROOT / 'test' / 'masks'

BATCH_SIZE = 8
LEARNING_RATE = 1e-4
EPOCHS = 100
NUM_WORKERS = 0
IMAGE_SIZE = (256, 256)
NUM_CLASSES = 1
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

Using device: cpu


#### Dataset Preparation

In [3]:
def dice_loss(pred, target, eps=1e-6):
    pred_prob = torch.sigmoid(pred)
    pred_flat = pred_prob.reshape(pred.shape[0], -1)
    target_flat = target.reshape(target.shape[0], -1).float()
    intersection = (pred_flat * target_flat).sum(dim=1)
    return 1.0 - ((2.0 * intersection + eps) / (pred_flat.sum(dim=1) + target_flat.sum(dim=1) + eps)).mean()


def tversky_loss(pred, target, alpha=0.5, beta=0.5, eps=1e-6):
    pred_prob = torch.sigmoid(pred)
    pred_flat = pred_prob.reshape(pred.shape[0], -1)
    target_flat = target.reshape(target.shape[0], -1).float()

    tp = (pred_flat * target_flat).sum(dim=1)
    fp = ((1.0 - target_flat) * pred_flat).sum(dim=1)
    fn = (target_flat * (1.0 - pred_flat)).sum(dim=1)

    tversky = (tp + eps) / (tp + alpha * fp + beta * fn + eps)
    return 1.0 - tversky.mean()


def bce_dice_tversky_loss(pred, target, bce_weight=1.0, dice_weight=1.0, tversky_weight=1.0):
    bce = nn.BCEWithLogitsLoss()
    return bce_weight * bce(pred, target) + dice_weight * dice_loss(pred, target) + tversky_weight * tversky_loss(pred, target)


def compute_iou_per_class(pred_logits, target, threshold=0.5):
    probs = torch.sigmoid(pred_logits)
    pred_mask = (probs >= threshold).float()
    target = target.float()
    metrics = {}

    for cls in [0, 1]:
        pred_cls = (pred_mask == cls).float()
        target_cls = (target == cls).float()
        intersection = (pred_cls * target_cls).sum().item()
        union = pred_cls.sum().item() + target_cls.sum().item() - intersection
        metrics[f'class_{cls}'] = (intersection / (union + 1e-6)) if union > 0 else 1.0

    return metrics


model = MobileUNETR(in_channels=3, num_classes=1).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)

print(model)


MobileUNETR(
  (encoder): Encoder(
    (stem): STEM(
      (conv1): ConvBlock(
        (conv): Conv2d(3, 32, kernel_size=(5, 5), stride=(2, 2), padding=(2, 2), bias=False)
        (bn): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (mn1): MobileNetv2(
        (block): Sequential(
          (0): Conv2d(32, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
          (2): SiLU(inplace=True)
          (3): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=128, bias=False)
          (4): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
          (5): SiLU(inplace=True)
          (6): Conv2d(128, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (7): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, t

In [4]:
class BRISCSegmentationDataset(Dataset):
    def __init__(self, image_dir, mask_dir, transform=None, target_transform=None):
        self.image_dir = Path(image_dir)
        self.mask_dir = Path(mask_dir)
        self.transform = transform
        self.target_transform = target_transform

        self.image_files = sorted(
            p for p in self.image_dir.iterdir() if p.is_file() and p.suffix.lower() in {'.jpg', '.jpeg', '.png'}
        )
        self.mask_files = {p.stem: p for p in self.mask_dir.iterdir() if p.is_file() and p.suffix.lower() == '.png'}

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        image_path = self.image_files[idx]
        mask_path = self.mask_files.get(image_path.stem)
        if mask_path is None:
            raise FileNotFoundError(f'Mask not found for image {image_path.name}')

        image = Image.open(image_path).convert('RGB')
        mask = Image.open(mask_path).convert('L')

        if self.transform:
            image = self.transform(image)
        if self.target_transform:
            mask = self.target_transform(mask)
        else:
            mask = np.array(mask, dtype=np.float32) / 255.0
            mask = torch.from_numpy(mask).unsqueeze(0)
            if image.shape[-2:] != mask.shape[-2:]:
                mask = torch.nn.functional.interpolate(mask.unsqueeze(0), size=image.shape[-2:], mode='nearest').squeeze(0)

        if image.shape[0] != 3:
            image = image[:3, :, :]

        return image, mask


def pad_to_size(image, target_size=(256, 256)):
    image = np.array(image)
    if image.ndim == 2:
        image = image[..., None]
    h, w = image.shape[:2]
    target_h, target_w = target_size
    pad_h = max(0, target_h - h)
    pad_w = max(0, target_w - w)
    if pad_h or pad_w:
        padded = np.zeros((target_h, target_w, image.shape[2]), dtype=image.dtype)
        padded[:h, :w, ...] = image
        return padded
    return image


def to_tensor(x):
    if isinstance(x, np.ndarray):
        x = torch.from_numpy(x)
    if isinstance(x, Image.Image):
        x = np.array(x)
        x = torch.from_numpy(x).permute(2, 0, 1)
    return x.float() / 255.0 if x.dtype == torch.float32 or x.dtype == torch.uint8 else x


def train_transform(image):
    image = np.array(image.resize(IMAGE_SIZE, resample=Image.BILINEAR))
    image = torch.from_numpy(image).permute(2, 0, 1).float() / 255.0
    return image


def mask_transform(mask):
    mask = np.array(mask.resize(IMAGE_SIZE, resample=Image.NEAREST), dtype=np.float32) / 255.0
    mask = torch.from_numpy(mask).unsqueeze(0)
    return mask


train_ds = BRISCSegmentationDataset(
    TRAIN_IMAGES,
    TRAIN_MASKS,
    transform=train_transform,
    target_transform=mask_transform,
)
val_ds = BRISCSegmentationDataset(
    VAL_IMAGES,
    VAL_MASKS,
    transform=train_transform,
    target_transform=mask_transform,
)

print(f'Train samples: {len(train_ds)}')
print(f'Validation samples: {len(val_ds)}')

sample_image, sample_mask = train_ds[0]
print('Image shape:', tuple(sample_image.shape))
print('Mask shape:', tuple(sample_mask.shape))
print('Mask min/max:', sample_mask.min().item(), sample_mask.max().item())

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)


Train samples: 3933
Validation samples: 860
Image shape: (3, 256, 256)
Mask shape: (1, 256, 256)
Mask min/max: 0.0 1.0


#### Training loop

In [ ]:
def train_one_epoch(model, loader, optimizer, device, log_every=10):
    model.train()
    running_loss = 0.0
    running_iou = {'class_0': 0.0, 'class_1': 0.0}
    total_seen = 0

    for batch_idx, (images, targets) in enumerate(loader, start=1):
        images = images.to(device, dtype=torch.float32)
        targets = targets.to(device, dtype=torch.float32)

        optimizer.zero_grad()
        logits = model(images)
        loss = bce_dice_tversky_loss(logits, targets)
        loss.backward()
        optimizer.step()

        batch_size = images.size(0)
        running_loss += loss.item() * batch_size
        total_seen += batch_size

        batch_iou = compute_iou_per_class(logits, targets)
        for key in running_iou:
            running_iou[key] += batch_iou[key] * batch_size

        if batch_idx % log_every == 0 or batch_idx == len(loader):
            avg_loss = running_loss / total_seen
            avg_iou = {key: running_iou[key] / total_seen for key in running_iou}
            print(
                f"  Batch {batch_idx}/{len(loader)} - loss={avg_loss:.4f} "
                f"- IoU[class_0]={avg_iou['class_0']:.4f} "
                f"- IoU[class_1]={avg_iou['class_1']:.4f}"
            )

    return running_loss / total_seen, {key: running_iou[key] / total_seen for key in running_iou}


def evaluate(model, loader, device):
    model.eval()
    running_loss = 0.0
    running_iou = {'class_0': 0.0, 'class_1': 0.0}
    total_seen = 0

    with torch.no_grad():
        for images, targets in loader:
            images = images.to(device, dtype=torch.float32)
            targets = targets.to(device, dtype=torch.float32)

            logits = model(images)
            loss = bce_dice_tversky_loss(logits, targets)

            batch_size = images.size(0)
            running_loss += loss.item() * batch_size
            total_seen += batch_size

            batch_iou = compute_iou_per_class(logits, targets)
            for key in running_iou:
                running_iou[key] += batch_iou[key] * batch_size

    return running_loss / total_seen, {key: running_iou[key] / total_seen for key in running_iou}


history = {
    "train_loss": [],
    "val_loss": [],
    "train_iou_class_0": [],
    "train_iou_class_1": [],
    "val_iou_class_0": [],
    "val_iou_class_1": [],
}

for epoch in tqdm(range(EPOCHS)):
    train_loss, train_iou = train_one_epoch(model, train_loader, optimizer, DEVICE, log_every=10)
    val_loss, val_iou = evaluate(model, val_loader, DEVICE)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_iou_class_0"].append(train_iou['class_0'])
    history["train_iou_class_1"].append(train_iou['class_1'])
    history["val_iou_class_0"].append(val_iou['class_0'])
    history["val_iou_class_1"].append(val_iou['class_1'])

    print(
        f"Epoch {epoch + 1}/{EPOCHS} - "
        f"train_loss={train_loss:.4f} - val_loss={val_loss:.4f} - "
        f"train_iou[class_0]={train_iou['class_0']:.4f} - train_iou[class_1]={train_iou['class_1']:.4f} - "
        f"val_iou[class_0]={val_iou['class_0']:.4f} - val_iou[class_1]={val_iou['class_1']:.4f}"
    )
    if val_loss < min(history["val_loss"]):
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "train_loss": train_loss,
            "val_loss": val_loss
        }, "MobileUnetR/checkpoints/best_model.pt")
    else:
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "train_loss": train_loss,
            "val_loss": val_loss
        }, "MobileUnetR/checkpoints/current_model.pt")


print("Training finished.")
print(history)


  0%|          | 0/100 [00:00<?, ?it/s]

  Batch 10/492 - loss=2.6992 - IoU[class_0]=0.3568 - IoU[class_1]=0.0185
  Batch 20/492 - loss=2.6915 - IoU[class_0]=0.3838 - IoU[class_1]=0.0191
  Batch 30/492 - loss=2.6861 - IoU[class_0]=0.4051 - IoU[class_1]=0.0187
  Batch 40/492 - loss=2.6760 - IoU[class_0]=0.4310 - IoU[class_1]=0.0196
  Batch 50/492 - loss=2.6628 - IoU[class_0]=0.4587 - IoU[class_1]=0.0214
  Batch 60/492 - loss=2.6467 - IoU[class_0]=0.4882 - IoU[class_1]=0.0242
  Batch 70/492 - loss=2.6335 - IoU[class_0]=0.5180 - IoU[class_1]=0.0263
  Batch 80/492 - loss=2.6190 - IoU[class_0]=0.5481 - IoU[class_1]=0.0298
  Batch 90/492 - loss=2.6065 - IoU[class_0]=0.5783 - IoU[class_1]=0.0344
  Batch 100/492 - loss=2.5988 - IoU[class_0]=0.6075 - IoU[class_1]=0.0380
  Batch 110/492 - loss=2.5907 - IoU[class_0]=0.6349 - IoU[class_1]=0.0421
  Batch 120/492 - loss=2.5827 - IoU[class_0]=0.6588 - IoU[class_1]=0.0474
  Batch 130/492 - loss=2.5740 - IoU[class_0]=0.6812 - IoU[class_1]=0.0569
  Batch 140/492 - loss=2.5659 - IoU[class_0]=0.